# BirdCLEF+ 2026: do synthetic clips help rare classes?

A short summary of a COMPX525 Deep Learning coursework project at the University of Waikato. The full code, report and per-class tables are on GitHub: [NecmiKural/birdclef-2026-audio-classification](https://github.com/NecmiKural/birdclef-2026-audio-classification).

**Question.** Can AudioLDM2 text-to-audio clips for rare classes improve classification? Two model families were each trained on the original data (A) and on data enriched with 150 synthetic clips (B):

- **BirdNET embeddings + ML:** BirdNET segment embeddings, pooled per clip (mean, std, max), standardised, then a one-vs-rest balanced SGD classifier.
- **CompactAudioCNN:** a small CNN trained from scratch on mel-spectrograms.

**Result.** The answer depends on the model family. From the README:

> BirdNET embeddings + ML reach 0.8940 validation macro ROC-AUC with synthetic enrichment and 0.8869 without, on one fixed 80/20 split of train_audio with one seed. This is not a Kaggle leaderboard score.

The from-scratch CNN went the other way: 0.8028 without synthetic clips and 0.7645 with them, on the same split.

## One clip, as the models see it

Up to the first 10 seconds of a recording of a rare class from the competition's `train_audio`, as a log-mel spectrogram. The audio stays on Kaggle; it is not redistributed.

In [ ]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt

# Kaggle has mounted competition data under both paths over time.
ROOT = next(p for p in (Path("/kaggle/input/competitions/birdclef-2026"), Path("/kaggle/input/birdclef-2026"))
            if (p / "train.csv").exists())
print("data:", ROOT, flush=True)
train = pd.read_csv(ROOT / "train.csv")
row = train[train["primary_label"].astype(str) == "116570"].iloc[0]  # Southern Spectacled Caiman
info = sf.info(ROOT / "train_audio" / row["filename"])
audio, sr = sf.read(ROOT / "train_audio" / row["filename"], frames=int(info.samplerate * 10), dtype="float32")
audio = audio.mean(axis=1) if audio.ndim > 1 else audio
print("clip:", row["filename"], sr, "Hz", round(len(audio) / sr, 1), "s", flush=True)
mel = librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128), ref=np.max)

fig, ax = plt.subplots(figsize=(10, 3.5))
librosa.display.specshow(mel, sr=sr, x_axis="time", y_axis="mel", ax=ax)
ax.set_title(f"{row['common_name']}: first {len(audio) / sr:.1f} s, log-mel")
plt.tight_layout()
plt.show()

## Rare classes: what the synthetic clips changed

Per-class validation ROC-AUC for the embedding pipeline, without (A1) and with (B1) synthetic clips, for the rare classes that got synthetic clips. The table below is copied from `docs/rare_class_a1_vs_b1.csv` at commit [`d6540d1`](https://github.com/NecmiKural/birdclef-2026-audio-classification/blob/d6540d14b8524af137806dcefdde460a79e9256f/docs/rare_class_a1_vs_b1.csv); the check fails if it stops matching the README.

In [ ]:
RARE_CSV = """
primary_label,common_name,class_name,train_csv_count,val_positive_rows,synth_train_clips,A1_AUC,B1_AUC,delta_B1_minus_A1
116570,Southern Spectacled Caiman,Reptilia,1,1,9,0.5,0.4352,-0.0648
516975,Hooded Capuchin,Mammalia,1,1,9,0.5,0.3121,-0.1879
23150,Central Dwarf Frog,Amphibia,1,1,9,0.5,0.4736,-0.0264
23724,Waxy Monkey Tree Frog,Amphibia,1,1,9,0.5,0.4582,-0.0418
209233,Feral Horse,Mammalia,2,0,8,,,
70711,Cei's White-lipped Frog,Amphibia,2,1,8,0.2994,0.4804,0.181
24321,Mato Grosso Snouted Tree Frog,Amphibia,2,1,8,0.771,0.3714,-0.3996
74580,Black-tailed Marmoset,Mammalia,3,1,7,0.9439,0.9886,0.0446
64898,Yungas de la Paz Poison Frog,Amphibia,3,1,7,0.7151,0.9884,0.2733
555123,Usina Tree Frog,Amphibia,3,1,7,0.9542,0.9864,0.0322
476521,Cuyaba Dwarf Frog,Amphibia,3,1,7,0.5627,0.8989,0.3361
25214,Muller's Termite Frog,Amphibia,3,1,7,0.9584,0.9482,-0.0102
sptnig1,Spot-tailed Nightjar,Aves,3,1,7,0.7084,0.9215,0.2131
23176,Cope's Swamp Froglet,Amphibia,4,1,6,0.9571,0.8458,-0.1113
23154,Bahia Dwarf Frog,Amphibia,5,1,5,0.9996,0.9452,-0.0544
66971,Paraguayan Swimming Frog,Amphibia,5,1,5,0.5003,0.9851,0.4848
738183,White-coated Titi,Mammalia,5,1,5,0.9968,0.9539,-0.0429
1595929,Uruguay Harlequin Frog,Amphibia,5,1,5,0.4868,0.9345,0.4477
22985,Peter's Jungle Frog,Amphibia,6,1,4,0.8992,0.4946,-0.4046
22961,Pointedbelly Frog,Amphibia,6,1,4,0.8912,0.9879,0.0967
67252,Milk Frog,Amphibia,6,1,4,0.993,0.9632,-0.0298
22930,Basin White-lipped Frog,Amphibia,6,1,4,0.9292,0.9923,0.0631
760266,Prionacris erosa,Insecta,7,1,3,0.9998,0.9687,-0.0311
22967,Marbled White-lipped Frog,Amphibia,8,2,2,0.999,0.9991,0.0001
555145,Polka-dot Tree Frog,Amphibia,9,2,1,0.9216,0.9157,-0.0059
"""
rare = pd.read_csv(io.StringIO(RARE_CSV.strip())).dropna(subset=["delta_B1_minus_A1"])
delta = rare["delta_B1_minus_A1"]
summary = dict(targets=len(rare), mean_delta=round(float(delta.mean()), 3),
               improved=int((delta > 0).sum()), worsened=int((delta < 0).sum()))
print(summary, flush=True)
# The README: "On its 24 rare-class targets the mean ΔAUC is +0.032, with 11 improved and 13 worsened"
assert summary == dict(targets=24, mean_delta=0.032, improved=11, worsened=13)

In [ ]:
r = rare.sort_values("delta_B1_minus_A1")
colours = ["#0d5e66" if d > 0 else "#c2410c" for d in r["delta_B1_minus_A1"]]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(r["common_name"], r["delta_B1_minus_A1"], color=colours)
for y, (d, n) in enumerate(zip(r["delta_B1_minus_A1"], r["val_positive_rows"])):
    ax.annotate(f"{n} val. pos.", (d, y), xytext=(4 if d > 0 else -4, 0), textcoords="offset points",
                ha="left" if d > 0 else "right", va="center", fontsize=8, color="#5a5446")
ax.axvline(0, color="#1d1b16", linewidth=1)
ax.set_xlabel("ROC-AUC change with synthetic clips (B1 minus A1)")
ax.set_title("Embedding pipeline, rare classes: 11 improved, 13 worsened")
plt.tight_layout()
plt.show()

## How far this goes

- One fixed 80/20 split of `train_audio` and one seed. Most rare classes have 1 to 4 positive validation rows, so each per-class change above rests on very few clips.
- Validation scores, not Kaggle leaderboard scores; the soundscapes were not used in the reported results.
- BirdNET and AudioLDM2 (`cvssp/audioldm2`) are CC BY-NC-SA 4.0, and the competition data is for non-commercial use under the competition rules. The project's own code is MIT licensed.

The report in the repository has the split, the methods, the network diagrams and the per-class results for both pipelines.